In [1]:
%load_ext autoreload
%autoreload 2

from tasks.autoencoder import AETask
import os
import torch
from tqdm import tqdm
import einx
from sentence_transformers import SentenceTransformer
from sentence_transformers.models import Pooling, Transformer, Normalize
from transformers import AutoModel, AutoTokenizer, T5TokenizerFast, AutoModelForCausalLM, T5Tokenizer
from datasets import load_dataset, load_from_disk
import re
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm
from data.datasets import WikipediaDataset, WikipediaDatasetConfig
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, models
import wandb
from tasks.dlclm import DLCLMTask
from mauve import compute_mauve, get_features_from_input
from model.encoder import HSEMHead, HSEMHeadConfig, SEMHeadConfig, SEMHead
import os
from data.datasets import FineWebDataset

/home/l/leog/links/scratch/sentence_diffusion/venv/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/l/leog/links/scratch/sentence_diffusion/venv/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assign

In [1]:
from sentence_transformers import SentenceTransformer

# Load the model
model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device='cuda')

# We recommend enabling flash_attention_2 for better acceleration and memory saving,
# together with setting `padding_side` to "left":
# model = SentenceTransformer(
#     "Qwen/Qwen3-Embedding-0.6B",
#     model_kwargs={"attn_implementation": "flash_attention_2", "device_map": "auto"},
#     tokenizer_kwargs={"padding_side": "left"},
# )

In [18]:
def get_detailed_instruct(task_description: str, query: str) -> str:
    return f'Instruct: {task_description}\nQuery:{query}'

In [23]:
model.prompts

{'query': 'Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:',
 'document': ''}

In [7]:
# The queries and documents to embed
queries = [
    "Then, I went to the garden to the garden to pick some cherries.",
    "Explain gravity.",
]
documents = [
    "Last, year, diego went to Africa to meet an old friend.",
    "Gravity is a force that attracts two bodies towards each other.",
    "I love going in the forest and foraging mushrooms!",
    "You love going in the forest and foraging mushrooms!",
    "Can you explain the maillard reaction?",
    "Explain why the sky is blue.",
    "Yesterday, I went to the store to buy some things."
]

#prompt = "Instruct: Given some text, retrieve other texts which share some high level attributes. This can include semantic information, type of text, intent, style. \nText:"
prompt = "Represent the text with high-level features. This can include semantic information, type of text, intent, style. Text: "
# prompt = "Represent only the information provided in this text; do not represent user requests: "
query_embeddings = model.encode(queries, prompt=prompt)
document_embeddings = model.encode(documents, prompt=prompt)

In [8]:
similarity = model.similarity(query_embeddings, document_embeddings)
similarity

tensor([[0.8175, 0.8386, 0.9061, 0.8952, 0.8412, 0.8487, 0.8798],
        [0.8261, 0.9366, 0.9186, 0.9336, 0.9719, 0.9821, 0.8365]])

In [12]:
torch.softmax(similarity/0.05,-1)

tensor([[0.0498, 0.0758, 0.2930, 0.2354, 0.0800, 0.0929, 0.1731],
        [0.0148, 0.1352, 0.0942, 0.1273, 0.2742, 0.3360, 0.0182]])

In [153]:
import torch

In [15]:
x = torch.IntTensor([[1,2,3],[4,5,6]])
prompt = [3,3,4,4]

In [22]:
[[] for i in range(10)]

[[], [], [], [], [], [], [], [], [], []]

In [18]:
torch.cat([x,y])

RuntimeError: Tensors must have same number of dimensions: got 2 and 1

In [7]:
[prompt + x]

TypeError: can only concatenate list (not "Tensor") to list